In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    TimestampType, IntegerType, StringType, DoubleType
)
import time
import uuid
import subprocess

spark = SparkSession.builder \
    .appName("AML Streaming Producer") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

aml_schema = StructType([
    StructField("Timestamp", TimestampType(), True),
    StructField("From Bank", IntegerType(), True),
    StructField("Account2", StringType(), True),
    StructField("To Bank", IntegerType(), True),
    StructField("Account4", StringType(), True),
    StructField("Amount Received", DoubleType(), True),
    StructField("Receiving Currency", StringType(), True),
    StructField("Amount Paid", DoubleType(), True),
    StructField("Payment Currency", StringType(), True),
    StructField("Payment Format", StringType(), True),
    StructField("Is Laundering", IntegerType(), True)
])

source_file = "hdfs://10.84.129.52:9000/aulas/fabricio_moreira/project/dataset/LI-Medium_Trans.csv"
stream_input_path = "hdfs://10.84.129.52:9000/aulas/fabricio_moreira/project/stream/input"
temp_path = "hdfs://10.84.129.52:9000/aulas/fabricio_moreira/project/stream/tmp_producer"


df = spark.read \
    .option("header", True) \
    .option("timestampFormat", "yyyy/MM/dd HH:mm") \
    .schema(aml_schema) \
    .csv(source_file)

print("Total records:", df.count())

In [ ]:
num_batches = 10
delay_seconds = 10

batches = df.randomSplit([1.0] * num_batches, seed=42)

hadoop_conf = spark._jsc.hadoopConfiguration()
fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(hadoop_conf)
Path = spark._jvm.org.apache.hadoop.fs.Path

# Criar pasta de input, se não existir
fs.mkdirs(Path(stream_input_path))

# Limpar pasta temporária do producer
fs.delete(Path(temp_path), True)

for i, batch_df in enumerate(batches, start=1):
    unique_id = str(uuid.uuid4())[:8]

    tmp_batch_dir = f"{temp_path}/batch_{i:03d}_{unique_id}"
    final_file = f"{stream_input_path}/batch_{i:03d}_{unique_id}.csv"

    print(f"Preparar batch {i}...")

    batch_df.coalesce(1) \
        .write \
        .mode("overwrite") \
        .option("header", True) \
        .csv(tmp_batch_dir)

    # Encontrar o part file criado pelo Spark
    files = fs.listStatus(Path(tmp_batch_dir))

    part_file = None
    for file in files:
        name = file.getPath().getName()
        if name.startswith("part-") and name.endswith(".csv"):
            part_file = file.getPath()
            break

    if part_file is None:
        raise Exception(f"Nenhum ficheiro part-*.csv encontrado em {tmp_batch_dir}")

    # Mover o part file para a pasta de input com nome final
    fs.rename(part_file, Path(final_file))

    # Remover pasta temporária do batch
    fs.delete(Path(tmp_batch_dir), True)

    print(f"Batch {i} enviado para {final_file}")

    time.sleep(delay_seconds)

print("Producer terminou.")